# Course 15 lab — Enterprise PDLC and CI/CD delivery controls

This notebook uses the same deterministic engine as the command-line lab. It separates proposal, validation, authorization, deployment, and runtime acceptance.

## 1. Load the trusted and untrusted fixtures
The control-plane registry is treated as trusted fixture data. The reference and candidate packages are untrusted proposals.

In [ ]:
import copy, importlib.util, pathlib, sys
root = pathlib.Path.cwd()
if not (root / 'lab.py').exists():
    root = root / 'curriculum/intermediate/05-enterprise-pdlc-ci-cd-integration/northstar-sdd-delivery'
spec = importlib.util.spec_from_file_location('course15_lab', root / 'lab.py')
lab = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = lab
spec.loader.exec_module(lab)
control = lab.load_control_plane()
reference = lab.load_reference()
candidate = lab.load_candidate()
reference['change_manifest']['change_id'], candidate['fixture_status']

## 2. Compare a finding-free proposal with an unsafe green PR
Eligibility is not authorization. The reference should stop at integration review; the unsafe candidate should be blocked even though conventional technical checks pass.

In [ ]:
reference_gates = lab.evaluate_delivery(reference, control)
candidate_gates = lab.evaluate_delivery(candidate, control)
{
    'reference': lab.aggregate(reference_gates),
    'candidate': lab.aggregate(candidate_gates),
    'candidate_technical_quality': next(g.status for g in candidate_gates if g.gate == 'technical_quality'),
    'candidate_findings': sorted(lab.finding_codes(candidate_gates)),
}

## 3. Inspect every gate, including non-applicable transitions
A complete gate set is important. A deployment gate that did not run should say `NOT_APPLICABLE`, not disappear or masquerade as `PASS`.

In [ ]:
[(gate.gate, gate.status, [item.code for item in gate.findings]) for gate in reference_gates]

## 4. Evidence is claim-specific and exact-subject
Change one commit binding. The evidence record still exists and still says `PASS`, but it no longer supports this PR.

In [ ]:
wrong_registry = copy.deepcopy(control)
wrong_registry['evidence_registry']['EVID-CLASS-001']['subject']['commit'] = 'old-commit'
sorted(lab.finding_codes(lab.evaluate_delivery(reference, wrong_registry)))

## 5. Merge queues require semantic revalidation
Git can merge files while requirements conflict. Model that independent failure without changing the implementation.

In [ ]:
queued = copy.deepcopy(reference)
queued['merge_queue']['semantic_conflicts'] = ['AI-2322 changes the same human-review invariant']
sorted(lab.finding_codes(lab.evaluate_delivery(queued, control)))

## 6. Authorization receipts are exact-subject and single-use
The gate can verify an integration request, but only the trusted receipt consumer changes receipt state. A second use must fail.

In [ ]:
integration = copy.deepcopy(reference)
registry = copy.deepcopy(control)
integration['integration'] = {'requested': True, 'receipt_id': 'INT-AI2310-001'}
before = lab.verify_receipt(integration, registry, 'INT-AI2310-001', kind='INTEGRATION')
consumed = lab.consume_receipt(integration, registry, 'INT-AI2310-001', kind='INTEGRATION')
try:
    lab.consume_receipt(integration, registry, 'INT-AI2310-001', kind='INTEGRATION')
    replay = 'unexpected success'
except ValueError as error:
    replay = str(error)
before, consumed['state'], replay

## 7. Prevent deployment artifact substitution
Approval of source is not approval of any artifact built later. The receipt, provenance subject, build record, and deployment request must name the same digest.

In [ ]:
deployment = copy.deepcopy(integration)
deployment['deployment'].update({'requested': True, 'receipt_id': 'DEP-AI2310-001', 'artifact_digest': 'sha256:substituted'})
sorted(lab.finding_codes(lab.evaluate_delivery(deployment, registry)))

## 8. Runtime zero needs a denominator and healthy instrumentation
Compare a healthy observation window with one that cannot support a conclusion.

In [ ]:
healthy = copy.deepcopy(reference)
healthy['runtime']['observed'] = True
blind = copy.deepcopy(healthy)
blind['runtime']['instrumentation_health'] = 'UNHEALTHY'
{
    'healthy': sorted(lab.finding_codes(lab.evaluate_delivery(healthy, control))),
    'blind': sorted(lab.finding_codes(lab.evaluate_delivery(blind, control))),
}

## 9. Run the bounded conformance and mutation suites
These results establish reproducibility for the teaching fixture, not general control effectiveness or production assurance.

In [ ]:
conformance = lab.run_conformance_suite()
evaluation = lab.run_evaluation()
assert conformance['passed'] == conformance['total'] == 11
assert evaluation['passed'] == evaluation['total'] == 68
conformance, evaluation

## 10. Production upgrade exercise
Replace fixture trust with authenticated identities, protected policy distribution, isolated untrusted execution, signed attestations, immutable artifact storage, and a transactional compare-and-swap receipt registry shared across processes and regions. Add replay tests across separate workers, auditable overrides, complete rollout-slice monitoring, delivery-gap detection, and monitored control-plane SLOs. Keep validation separate from authorization.